# Notebook B of 2. Quantum kernels and evaluation for the seed study (CPU is fine)

**What this notebook does.** It reads the raw encoder embeddings written by Notebook A (`seeds_A_embeddings.ipynb`)
and runs everything after the encoder, exactly as the main run did: PCA to 8 dimensions, scaling to [0, pi], the
1000/200 draw, the IQP quantum kernel, the three classical SVMs, the utility gap with its bootstrap interval, and
the alignments. No GPU and no downloads are needed.

**Order of work.**
1. Section 3 reproduces seed 42 from its raw file and compares the features, and three rows of the quantum kernel,
   with what the original run saved on Drive. Every configuration must print `PASS` before Section 2 is trusted.
2. Section 4 runs seeds 7 and 123. Each (configuration, seed) takes well under a minute.
3. Section 5 prints the three-seed summary used in the paper.

**Inputs.** `<tag>_seed<seed>_raw.npz` from Notebook A; for the reproduction check, the seed-42 feature and kernel
files from the original run (already in `qml_project`). **Outputs.** `<tag>_seed<seed>_grid_record.json` for seeds 7
and 123, plus the matching kernel file. Upload the 12 JSON records.

**Contents.**
0. Setup and inventory of input files
1. Preprocessing (PCA, scaling, the 1000/200 draw)
2. Quantum kernel and its self-check
3. Seed-42 reproduction check (must pass)
4. Seeds 7 and 123
5. Three-seed summary
6. Notes and troubleshooting

---
# Section 0. Setup and inventory

In [ ]:
!pip install pennylane pennylane-lightning scikit-learn numpy --quiet
import numpy as np, os, json, time, warnings
import pennylane as qml, sklearn
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics.pairwise import rbf_kernel
warnings.filterwarnings('ignore')
print('scikit-learn', sklearn.__version__, '| pennylane', qml.__version__)

USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/qml_project'
else:
    SAVE_DIR = 'qml_project'
os.makedirs(SAVE_DIR, exist_ok=True)
def find_file(f):
    for p in [os.path.join(SAVE_DIR, f), f]:
        if os.path.exists(p): return p
    return None

N_QUBITS = 8
TAGS = ['sst2_bert', 'agnews_roberta', 'cifar10', 'fashion_vit', 'mscoco_flava', 'snlive_siglip']
META = {'sst2_bert': ('text', 'BERT', 'binary'), 'agnews_roberta': ('text', 'RoBERTa', 'macro'),
        'cifar10': ('vision', 'CLIP', 'binary'), 'fashion_vit': ('vision', 'ViT', 'binary'),
        'mscoco_flava': ('multimodal', 'FLAVA', 'binary'), 'snlive_siglip': ('multimodal', 'SigLIP', 'macro')}
NO_SUBSAMPLE = {'fashion_vit'}   # its pool is already exactly 1000/200 in the original notebook

print('\nInput inventory (raw embeddings from Notebook A):')
for tag in TAGS:
    print(f'  {tag:<16}', ' '.join(f'seed {s}: {"ok" if find_file(f"{tag}_seed{s}_raw.npz") else "MISSING"}' for s in (42, 7, 123)))

---
# Section 1. Preprocessing (identical in every original notebook)

PCA to 8 components fitted on the training pool, min-max scaling to [0, pi] fitted on the training pool with test
values clipped, then the 1000-of-pool and 200-of-pool draws with two separate `RandomState(seed)` generators.

In [ ]:
def features(tag, seed):
    z = np.load(find_file(f'{tag}_seed{seed}_raw.npz'))
    E_tr, y_tr, E_te, y_te = z['E_tr'], z['y_tr'], z['E_te'], z['y_te']
    pca = PCA(n_components=N_QUBITS, random_state=seed).fit(E_tr)
    P_tr, P_te = pca.transform(E_tr), pca.transform(E_te)
    sc = MinMaxScaler(feature_range=(0, np.pi)).fit(P_tr)
    X_tr, X_te = np.clip(sc.transform(P_tr), 0, np.pi), np.clip(sc.transform(P_te), 0, np.pi)
    if tag in NO_SUBSAMPLE: return X_tr, y_tr, X_te, y_te
    i_tr = np.random.RandomState(seed).choice(len(X_tr), 1000, replace=False)
    i_te = np.random.RandomState(seed).choice(len(X_te), 200, replace=False)
    return X_tr[i_tr], y_tr[i_tr], X_te[i_te], y_te[i_te]

---
# Section 2. Quantum kernel (identical circuit to every original notebook)

Single-layer IQP map on 8 qubits, kernel = probability of the all-zeros outcome after U(x) then U(x') adjoint, exact
statevector simulation on `lightning.qubit`. The originals evaluated one circuit per pair of points (about 80 minutes
per configuration). Since K(x, x') = |<phi(x')|phi(x)>|^2 with |phi(x)> = U(x)|0>, `fast_kernel` simulates each
point's state once and forms all entries by one matrix product, which is identical on an exact simulator. It was
checked against the saved seed-42 SST-2 kernel (all 1,000,000 training entries agree to 1e-15; quantum F1 unchanged),
and the cell below re-checks it against the per-pair circuit on every run.

In [ ]:
dev = qml.device('lightning.qubit', wires=N_QUBITS)
def iqp_feature_map(x, wires):
    n = len(wires)
    for w in wires: qml.Hadamard(wires=w)
    for i, w in enumerate(wires): qml.RZ(x[i], wires=w)
    for i in range(n - 1):
        qml.CNOT(wires=[wires[i], wires[i+1]]); qml.RZ(x[i] * x[i+1], wires=wires[i+1]); qml.CNOT(wires=[wires[i], wires[i+1]])

@qml.qnode(dev)
def kernel_circuit(x1, x2):
    iqp_feature_map(x1, wires=range(N_QUBITS)); qml.adjoint(iqp_feature_map)(x2, wires=range(N_QUBITS))
    return qml.probs(wires=range(N_QUBITS))
quantum_kernel = lambda a, b: float(kernel_circuit(a, b)[0])     # the original per-pair kernel

@qml.qnode(dev)
def feature_state(x):
    iqp_feature_map(x, wires=range(N_QUBITS)); return qml.state()
def fast_kernel(A, B=None):
    SA = np.array([feature_state(a) for a in A]); SB = SA if B is None else np.array([feature_state(b) for b in B])
    return np.abs(SA.conj() @ SB.T) ** 2

def kta(K, y):
    y = np.asarray(y); n = len(y); Y = np.zeros((n, n))
    for c in np.unique(y): m = (y == c).astype(float); Y += np.outer(m, m)
    H = np.eye(n) - np.ones((n, n)) / n; Kc, Yc = H @ K @ H, H @ Y @ H
    return float(np.sum(Kc * Yc) / np.sqrt(np.sum(Kc * Kc) * np.sum(Yc * Yc)))
def concentration(K):
    mu = np.maximum(np.linalg.eigvalsh(K), 0); lam = mu / mu.sum()
    return float(np.exp(-np.sum(lam * np.log(lam + 1e-15))) / K.shape[0])

_A = np.random.RandomState(1).uniform(0, np.pi, (20, N_QUBITS))
_slow = np.array([[quantum_kernel(_A[i], _A[j]) for j in range(20)] for i in range(20)])
assert np.allclose(np.diag(_slow), 1.0) and np.allclose(_slow, _slow.T)
assert np.abs(fast_kernel(_A) - _slow).max() < 1e-10
print('kernel self-check passed: fast kernel equals the per-pair circuit')

---
# Section 3. Seed-42 reproduction check (required)

Rebuilds the seed-42 features from Notebook A's raw file and compares them with the features the original run saved,
then recomputes three test-kernel rows with the original per-pair circuit and compares them with the saved kernel.
The deciding test is the kernel rows. `PASS` means the recomputed kernel entries agree with the saved ones within
2e-3; GPU arithmetic typically gives differences near 1e-5, while a real pipeline difference gives 1e-2 or more. `MIRRORED` means some PCA columns
came out as pi minus the original, a scikit-learn version difference that would change the kernel; see the notes at
the end. `FAIL` means the pipeline does not reproduce the original; stop and report it.

What a passing configuration looks like:
```
=== sst2_bert ===
  train features vs kernel_matrices_week3.npz: PASS (max abs diff 1.56e-05)
  test features: PASS
  3 test-kernel rows vs saved kernel: max abs diff 5.44e-06 (within tolerance 2e-03)
  sst2_bert: PASS
```

In [ ]:
REFERENCE_FILES = {
    'sst2_bert':      ['kernel_matrices_week3.npz', 'embeddings_week2.npz'],
    'agnews_roberta': ['agnews_roberta_embeddings.npz', 'agnews_roberta_kernel_matrices.npz'],
    'cifar10':        ['cifar10_embeddings.npz', 'cifar10_kernel_matrices.npz'],
    'fashion_vit':    ['fashion_vit_embeddings.npz', 'fashion_vit_kernel_matrices.npz'],
    'mscoco_flava':   ['mscoco_flava_embeddings.npz', 'mscoco_flava_kernel_matrices.npz'],
    'snlive_siglip':  ['snlive_siglip_embeddings.npz', 'snlive_siglip_kernel_matrices.npz']}
KERNEL_FILES = {'sst2_bert': 'kernel_matrices_week3.npz'}
TEST_REFERENCE = {'sst2_bert': 'sst2_bert_test_labels.npz'}   # SST-2 test features recovered by sst2_bert_complete.ipynb

def load_reference(tag):
    for f in REFERENCE_FILES[tag]:
        p = find_file(f)
        if p is None: continue
        z = np.load(p)
        for a, b in [('X_tr_subset', 'X_te_subset'), ('X_tr', 'X_te')]:
            if a in z.files:
                R_te = z[b] if b in z.files else None
                if R_te is None and tag in TEST_REFERENCE and find_file(TEST_REFERENCE[tag]):
                    R_te = np.load(find_file(TEST_REFERENCE[tag]))['X_te']
                return f, z[a], R_te
    return None, None, None

FEATURE_TOL, KERNEL_TOL = 2e-3, 2e-3   # above float noise, far below any real pipeline difference

def compare(X, R, tol=FEATURE_TOL):
    d = np.abs(X - R).max(axis=0); dm = np.abs(X - (np.pi - R)).max(axis=0)
    if (d < tol).all(): return 'PASS', float(d.max())
    if ((d < tol) | (dm < tol)).all(): return 'MIRRORED', int((dm < tol).sum())
    return 'FAIL', float(d.max())

check = {}
for tag in TAGS:
    print(f'\n=== {tag} ===')
    if not find_file(f'{tag}_seed42_raw.npz'): print('  seed-42 raw file missing; run Notebook A'); check[tag] = 'NO RAW'; continue
    ref, R_tr, R_te = load_reference(tag)
    if ref is None: print('  no seed-42 reference in Drive'); check[tag] = 'NO REFERENCE'; continue
    X_tr, y_tr, X_te, y_te = features(tag, 42)
    st, d = compare(X_tr, R_tr)
    print(f'  train features vs {ref}: {st}', f'({d} of 8 columns mirrored)' if st == 'MIRRORED' else f'(max abs diff {d:.2e})')
    if R_te is not None:
        st2, d2 = compare(X_te, R_te); print(f'  test features: {st2} (max abs diff {d2:.2e})' if st2 != 'MIRRORED' else f'  test features: MIRRORED')
        if st == 'PASS' and st2 != 'PASS': st = st2
    # the deciding test: does this pipeline reproduce the saved quantum kernel?
    kf = find_file(KERNEL_FILES.get(tag, f'{tag}_kernel_matrices.npz'))
    if kf and R_te is not None and st != 'MIRRORED':
        K = np.load(kf)['K_test_quantum']
        rows = np.array([[quantum_kernel(X_te[i], X_tr[j]) for j in range(len(X_tr))] for i in range(3)])
        kd = np.abs(rows - K[:3]).max()
        print(f'  3 test-kernel rows vs saved kernel: max abs diff {kd:.2e} ({"within" if kd < KERNEL_TOL else "beyond"} tolerance {KERNEL_TOL:.0e})')
        st = 'PASS' if kd < KERNEL_TOL else 'FAIL'
    check[tag] = st; print(f'  {tag}: {st}')
print('\nSUMMARY'); [print(f'  {t:<16} {s}') for t, s in check.items()]
print('\nAll PASS. Section 4 may be run.' if all(v == 'PASS' for v in check.values()) else '\nResolve the non-PASS configurations first.')

---
# Section 4. Seeds 7 and 123

Only configurations that passed Section 3 are run. Existing records are skipped and kernels are checkpointed, so the
cell can be rerun safely. Each line printed is one finished (configuration, seed) with its gap, bootstrap interval,
quantum and best classical F1, and classical alignment.

In [ ]:
def run_pair(tag, seed):
    domain, encoder, metric = META[tag]; avg = 'binary' if metric == 'binary' else 'macro'
    f1 = lambda a, b: float(f1_score(a, b, average=avg))
    X_tr, y_tr, X_te, y_te = features(tag, seed)
    assert X_tr.shape == (1000, N_QUBITS) and X_te.shape == (200, N_QUBITS)
    base, fitted = {}, {}
    for name, clf in [('Linear', SVC(kernel='linear', C=1.0, random_state=seed)),
                      ('RBF', SVC(kernel='rbf', C=1.0, gamma='scale', random_state=seed)),
                      ('Polynomial', SVC(kernel='poly', degree=3, coef0=1, C=1.0, gamma='scale', random_state=seed))]:
        yp = clf.fit(X_tr, y_tr).predict(X_te); fitted[name] = clf
        base[name] = {'accuracy': float(accuracy_score(y_te, yp)), 'f1': f1(y_te, yp)}
    best = max(base, key=lambda k: base[k]['f1']); best_f1 = base[best]['f1']
    kf = os.path.join(SAVE_DIR, f'{tag}_seed{seed}_kernel_matrices.npz')
    if os.path.exists(kf):
        z = np.load(kf); K_tr, K_te = z['K_train_quantum'], z['K_test_quantum']
    else:
        K_tr, K_te = fast_kernel(X_tr), fast_kernel(X_te, X_tr)
        np.savez(kf, K_train_quantum=K_tr, K_test_quantum=K_te, X_tr=X_tr, y_tr=y_tr, X_te=X_te, y_te=y_te)
    y_q = SVC(kernel='precomputed', C=1.0).fit(K_tr, y_tr).predict(K_te); f1_q = f1(y_te, y_q); gap = f1_q - best_f1
    rng = np.random.RandomState(seed); y_c = fitted[best].predict(X_te); boot = []
    for _ in range(1000):
        idx = rng.randint(0, len(y_te), len(y_te))
        if len(np.unique(y_te[idx])) < 2: continue
        boot.append(f1(y_te[idx], y_q[idx]) - f1(y_te[idx], y_c[idx]))
    lo, hi = np.percentile(boot, [2.5, 97.5])
    K_rbf = rbf_kernel(X_tr, X_tr, gamma=1.0 / X_tr.shape[1]); off = ~np.eye(len(X_tr), dtype=bool)
    rec = {'tag': tag, 'seed': seed, 'domain': domain, 'encoder': encoder, 'metric': metric, 'n_qubits': N_QUBITS,
           'n_train': len(y_tr), 'n_test': len(y_te), 'quantum_accuracy': float(accuracy_score(y_te, y_q)),
           'quantum_f1': f1_q, 'best_classical_f1': best_f1, 'best_classical_kernel': best, 'gap': gap,
           'ci_lo': float(lo), 'ci_hi': float(hi), 'excludes_zero': bool(lo > 0 or hi < 0),
           'kta': kta(K_tr, y_tr), 'c_kta': kta(K_rbf, y_tr), 'concentration': concentration(K_tr),
           'offdiag_quantum': float(K_tr[off].mean()), 'offdiag_rbf': float(K_rbf[off].mean()),
           'classical_baselines': base, 'sklearn_version': sklearn.__version__,
           'pipeline': 'identical to seed-42 run; seed substituted for 42; embeddings from seeds_A_embeddings.ipynb'}
    json.dump(rec, open(os.path.join(SAVE_DIR, f'{tag}_seed{seed}_grid_record.json'), 'w'), indent=2)
    print(f'  {tag} seed {seed}: gap {gap:+.4f} [{lo:+.4f}, {hi:+.4f}] | quantum F1 {f1_q:.4f} vs {best} {best_f1:.4f} | A_C {rec["c_kta"]:.4f}')
    return rec

for tag in TAGS:
    if check.get(tag) != 'PASS': print(f'{tag}: skipped (Section 3 status {check.get(tag)})'); continue
    for seed in [7, 123]:
        if find_file(f'{tag}_seed{seed}_grid_record.json'): print(f'{tag} seed {seed}: record exists'); continue
        run_pair(tag, seed)
done = [(t, s) for t in TAGS for s in (7, 123) if find_file(f'{t}_seed{s}_grid_record.json')]
print(f'\n{len(done)} of 12 records on Drive')

---
# Section 5. Three-seed summary

Seed-42 values are copied from the paper's canonical results file (`grid_split.json`). Seeds 7 and 123 come from
the records written above.

In [ ]:
SEED42 = {"agnews_roberta": {"gap": 0.000214, "c_kta": 0.380437, "kta": 0.319909}, "cifar10": {"gap": 0.0, "c_kta": 0.708294, "kta": 0.272327}, "snlive_siglip": {"gap": -0.048772, "c_kta": 0.003499, "kta": 0.009206}, "sst2_bert": {"gap": -0.0514, "c_kta": 0.038991, "kta": 0.02118}, "fashion_vit": {"gap": -0.005127, "c_kta": 0.652706, "kta": 0.249212}, "mscoco_flava": {"gap": -0.059575, "c_kta": 0.018308, "kta": 0.043153}}
print(f"{'tag':<16} {'gap mean':>9} {'gap sd':>7} {'A_C mean':>9} {'A_C sd':>7}   gaps (42, 7, 123)")
sds = []
for tag in TAGS:
    recs = [SEED42[tag]] + [json.load(open(p)) for p in [find_file(f'{tag}_seed{s}_grid_record.json') for s in (7, 123)] if p]
    if len(recs) < 3: print(f'{tag:<16} {len(recs)} of 3 seeds available'); continue
    g = np.array([r['gap'] for r in recs]); a = np.array([r['c_kta'] for r in recs]); sds.append(g.std(ddof=1))
    print(f'{tag:<16} {g.mean():>+9.3f} {g.std(ddof=1):>7.3f} {a.mean():>9.3f} {a.std(ddof=1):>7.3f}   {[round(float(x), 3) for x in g]}')
if sds: print(f'\nmean gap sd across configurations: {np.mean(sds):.4f}')

---
# Section 6. Notes and troubleshooting

**MIRRORED in Section 3.** Newer scikit-learn changed the PCA sign convention, so a component x can become pi - x
after scaling, which changes the IQP kernel. Install the scikit-learn version used by the original run and rerun
Section 3. Do not run Section 4 on mirrored features.

**FAIL or NO REFERENCE.** NO REFERENCE means the original seed-42 feature file is not in `qml_project`; copy it there.
NO RAW means Notebook A has not produced that file yet. FAIL means a dataset or library changed since the main run;
report the printed difference instead of proceeding.

**Fixed pools.** For `cifar10`, `mscoco_flava`, and `snlive_siglip` the original pipelines take a fixed pool of
examples in dataset order, so seeds resample within that pool rather than drawing new examples. Their seed spread
is a lower bound on full resampling variance, and the paper says so wherever these numbers appear.

**Runtime statement in the paper.** The 83-minute figure in the paper is the cost of evaluating one circuit per pair
of points, as on hardware, where states cannot be read out and each overlap must be estimated by measurement. On a
statevector simulator the state-overlap shortcut used here gives the same kernel in seconds.